# Notebook 04 — Global Cross-Source Leakage and Conditional Split Freeze

**One CPU Run All** across verified D1, D2, D3 and D4 candidate text sources. Inputs are the **five D1 constituent CSVs**, the **four nonredundant D2 original/rewrite JSONs**, the **two D3 seed/variant JSONs**, and the **two D4 language CSVs** — **13 sources in total**. D2's 12,300-row categorized mirror is intentionally excluded as redundant; D5 is an authorship-feature archive, not primary phishing text.

What runs automatically:
1. Python, OS, core package versions, CPUs, RAM, disk, optional GPU inventory and self-tests.
2. SHA256 + size verification against the executed prior notebook receipts, reuse of verified attached/private inputs, or capped official-source download.
3. Strict normalized subject/body matching and candidate **body-only** and **coarse template** overlaps. Text hashes remain solely in RAM and are **never exported**.
4. D2 `No.` and D3 `Original_ID` family-aware connected-component union for exact matches, including cross-source overlaps, with counts of potential contamination components and comparable three-class label-conflict candidates (only for comparable three-class annotations).
5. Memory/download/elapsed-time receipts and a readable GO/NO-GO report.

**Fail closed:** Dataset rights, label semantics, human pair adjudication and frozen holdout design remain pending. The notebook **will not create a train/test split**, train a model, or export per-record cluster assignments on this run. A valid descriptive lineage result is **not** a split-freeze GO. Semantic near-duplicates, campaign/source lineage, and non-exact template variants need additional review.

**Files:** Safe aggregate reports only under `/kaggle/working/phishing_global_lineage_evidence/`. Downloaded private email content stays in `/kaggle/temp/phishing_global_lineage_private/` on Kaggle; do not publish it or any attached raw inputs. Network is needed only if pinned sources are not attached/cached. CPU recommended; no GPU quota needed.

In [ ]:
# NOTEBOOK 04: GLOBAL LINEAGE / DUPLICATE AUDIT — CPU, no raw content in public output
import csv, hashlib, importlib.metadata, json, os, platform, re, shutil
import subprocess, sys, tempfile, time, urllib.parse, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations

RELEASE="N04-GLOBAL-LINEAGE-v1.1-2026-10-31"
REV2="d13e9bb73d03f458e6d5b6cb6c7b0a708ea47e3d"
REV3="1e21dd4edbe5c64694f156bf5318c97c7c80681c"
REV4="d717e54"
KAGGLE=Path("/kaggle").is_dir()
TEMP=(Path("/kaggle/temp") if KAGGLE else Path(tempfile.gettempdir()))/"phishing_global_lineage_private"
OUT=(Path("/kaggle/working") if KAGGLE else Path.cwd())/"phishing_global_lineage_evidence"
TEMP.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
csv.field_size_limit(min(sys.maxsize,2**31-1))

# Name, family, path, file SHA256, exact bytes, source repo revision or official record.
SOURCES=[
 ("D1_Nazario","D1","Nazario.csv","b8fbc4158fbdfaff1ed98584c43d72e283c6352d7ade4d457d34c1d79488d184",7811841,"zenodo"),
 ("D1_Nazario5","D1","Nazario_5.csv","afef69828165619ced6661ca14a95444a1db82c7b25ae9c2619bbbb108844a44",11755517,"zenodo"),
 ("D1_SpamAssasin","D1","SpamAssasin.csv","3bfe8f8abff89f69a98456be50413c2fcb20a476141116dd84ee1507b980c00e",14879195,"zenodo"),
 ("D1_Enron","D1","Enron.csv","11f28a5bdd3603f1829681ab6b55071b5f9af781b4e6694e39216b2996a96883",44837129,"zenodo"),
 ("D1_CEAS08","D1","CEAS_08.csv","22375e7d5f5a8229dbe987914ee9b3705656c590038662a7df6054629b376074",67902056,"zenodo"),
 ("D2_original","D2","1_DataSet/cleaned_emails.json","b48026c91e8db399ea321d20e7e709f538aff685b9073d1a312ec09bee2e6d9c",7370034,"github2"),
 ("D2_deepseek","D2","3_RePhrase/DeepSeek/emails_rephrased_clean_deepseek.json","352dbd0fb7e065e819fb126e704b60d3c574f0ab8ea46ddb48306963d3787e0a",7528880,"github2"),
 ("D2_gpt4o","D2","3_RePhrase/GPT4/emails_rephrased_clean_openai4o.json","caac542ff660e5003eab68b908133ea42d9947f67284435ded9ac34b6a77b794",7079916,"github2"),
 ("D2_randomapi","D2","3_RePhrase/RandomAPI/emails_rephrased_openrouter_clean.json","2568744267134c722ad247f937deb154aad44cc90e85a5ccadb652c3d1cd71d7",7124691,"github2"),
 ("D3_seeds","D3","PhishFuzzer_emails_original_seed_v1.json","394d542e27dcef57321bc69f0f93b03e5965531f616eb3d3c79416cc60a30765",8218997,"github3"),
 ("D3_variants","D3","PhishFuzzer_emails_entity_rephrased_v1.json","62f9c1231cf7e9d1b4a9740680aefcf8723e689ac60ae171ea4e0eef70f1275d",48159145,"github3"),
 ("D4_chinese","D4","CN_Phishing_Email_dataset.csv","102e2c44239698c73e5e34abe277ca7aca25424cacb8af9425549a30fb0fc529",0,"hf"),
 ("D4_english","D4","GPT_Phishing_Email_dataset.csv","e3a335c8acacec09afaa4b9a14c7010195063310e4dcfbae38c1fa50b8d48dc3",0,"hf"),
]
CAPS={"D4_chinese":7_000_000,"D4_english":9_000_000}

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda:f.read(1024*1024),b""):h.update(part)
    return h.hexdigest()

def source_url(row):
    name,fam,path,sha,bytes_,kind=row
    p=urllib.parse.quote(path,safe="/")
    if kind=="zenodo": return "https://zenodo.org/records/8339691/files/"+p+"?download=1"
    if kind=="github2":return "https://raw.githubusercontent.com/HarmJ0y/PhishingSpamDataSet/"+REV2+"/"+p
    if kind=="github3":return "https://raw.githubusercontent.com/DataPhish/PhishFuzzer/"+REV3+"/"+p
    return "https://huggingface.co/datasets/Dizzzy0x00/LLMGen-Phishing-Email-Dataset/resolve/"+REV4+"/"+p+"?download=true"

def verified(path,row):
    if not path.is_file():return False
    size=row[4]; actual=path.stat().st_size
    if actual<1 or actual>CAPS.get(row[0],size+1024):return False
    if size and actual!=size:return False
    return sha_file(path)==row[3]

def retrieve(row):
    key,family,name,sha,size,kind=row
    filename=Path(name).name
    prior=[TEMP/filename,
        Path("/kaggle/temp/phishing_d1_private")/filename,
        Path("/kaggle/temp/phishing_d2_d5_private")/filename,
        Path("/kaggle/temp/phishing_pair_private")/filename,
        Path("/kaggle/working/AI-Generated-Phishing-Detection/data/datasets/D1_curated_emails/raw")/filename]
    inp=Path("/kaggle/input")
    if inp.is_dir():prior.extend(sorted(inp.rglob(filename)))
    for p in prior:
        if p.is_file() and verified(p,row):return p,"VERIFIED_CACHED_OR_ATTACHED",0
    dst=TEMP/filename;tmp=TEMP/(filename+".partial")
    cap=CAPS.get(key,size+1024)
    if shutil.disk_usage(TEMP).free < max(300_000_000,2*cap):
        raise ValueError("INSUFFICIENT_DISK")
    written=0
    try:
        req=urllib.request.Request(source_url(row),headers={"User-Agent":"AcademicGlobalLineageAudit/1.0"})
        with urllib.request.urlopen(req,timeout=180) as response:
            declared=response.headers.get("Content-Length")
            if declared and int(declared)>cap:raise ValueError("REMOTE_SIZE_TOO_LARGE")
            with tmp.open("wb") as target:
                while True:
                    part=response.read(1024*1024)
                    if not part:break
                    written+=len(part)
                    if written>cap:raise ValueError("DOWNLOAD_SIZE_CAP")
                    target.write(part)
        if not verified(tmp,row):raise ValueError("PINNED_SHA256_OR_SIZE_MISMATCH")
        tmp.replace(dst)
    finally:
        tmp.unlink(missing_ok=True)
    return dst,"OFFICIAL_VERIFIED_DOWNLOAD",written

def field(r,*names):
    keys={re.sub(r"[^a-z0-9]","",str(k).casefold()):v for k,v in r.items()}
    for key in names:
        k=re.sub(r"[^a-z0-9]","",key.casefold())
        if k in keys:return keys[k]
    return None

def rows_for(path):
    if path.suffix==".csv":
        with path.open("r",encoding="utf-8-sig",newline="") as f:
            reader=csv.DictReader(f,strict=True)
            header=reader.fieldnames or []
            if not header or len(header)!=len(set(header)):
                raise ValueError("INVALID_CSV_HEADER")
            for row in reader:
                if None in row or any(x is None for x in row.values()):
                    raise ValueError("MALFORMED_CSV_ROW")
                yield row
        return
    with path.open("r",encoding="utf-8-sig") as f:
        obj=json.load(f)
    if isinstance(obj,dict):
        arr=None
        for k in ("emails","data","records","items","dataset","results"):
            if isinstance(obj.get(k),list):arr=obj[k];break
        if arr is None and obj and all(isinstance(x,dict) for x in obj.values()):
            arr=list(obj.values())
        obj=arr
    if not isinstance(obj,list) or not obj:
        raise ValueError("UNSUPPORTED_JSON_CONTAINER")
    for row in obj:
        if not isinstance(row,dict):raise ValueError("NON_OBJECT_JSON_RECORD")
        yield row

def norm(x):
    return " ".join(str(x or "").casefold().split())

def sha_str(x):
    return hashlib.sha256(x.encode("utf-8")).hexdigest()

def text_keys(row):
    subject=norm(field(row,"subject","email_subject"))
    body=norm(field(row,"body","email_body","content","email_text","text","message"))
    if not subject and not body:return None,None,None
    strict=sha_str(json.dumps([subject,body],ensure_ascii=False))
    body_key=sha_str(body) if body else None
    # A coarse candidate template fingerprint, NOT a proven semantic near-duplicate.
    combined=(subject+"\n"+body)
    combined=re.sub(r"(?:https?://|www\.)\S+"," URL ",combined)
    combined=re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b"," EMAIL ",combined)
    combined=re.sub(r"\b\d+\b"," NUM ",combined)
    combined=" ".join(re.sub(r"[^\w\s]"," ",combined).split())
    coarse=sha_str(combined) if len(combined)>=120 else None
    return strict,body_key,coarse

def category(row,family):
    val=field(row,"Type","label","true_category","true_label")
    if family in ("D2","D3"):
        return val if isinstance(val,str) and val in ("Phishing","Spam","Valid") else "<unknown>"
    return str(val) if str(val) in ("0","1") else "<unknown>"

class DSU:
    def __init__(self):self.p={}
    def find(self,x):
        p=self.p
        if x not in p:p[x]=x
        while p[x]!=x:
            p[x]=p[p[x]];x=p[x]
        return x
    def union(self,a,b):
        aa,bb=self.find(a),self.find(b)
        if aa!=bb:self.p[max(aa,bb)]=min(aa,bb)

def parent_key(row,source,n):
    if source.startswith("D2_"):
        x=field(row,"No.")
        if isinstance(x,(str,int)) and str(x).strip():
            return "D2-PARENT|"+str(x).strip(),True
    if source.startswith("D3_"):
        x=field(row,"Original_ID")
        if isinstance(x,(str,int)) and str(x).strip():
            return "D3-PARENT|"+str(x).strip(),True
    return source+"|RECORD|"+str(n),False

def selftest():
    a={"Subject":"Greeting","Body":"go https://domain.invalid","Type":"Phishing","No.":7}
    b={"Subject":" greeting ","Body":" go  https://domain.invalid ","Type":"Phishing","No.":7}
    assert text_keys(a)[0]==text_keys(b)[0]
    assert category(a,"D2")=="Phishing"
    assert parent_key(a,"D2_original",1)==("D2-PARENT|7",True)
    d=DSU();d.union("a","b");d.union("b","c");assert d.find("a")==d.find("c")
    assert text_keys({"body":"x"})[0] and text_keys({"Body":""})[0] is None
    return "PASS"

def environment():
    pkg={}
    for name in ("numpy","pandas","scipy","torch","transformers","datasets","scikit-learn"):
        try:pkg[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:pkg[name]="not_installed"
    mem={}
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith(("MemTotal:","MemAvailable:")):
                    k,n,*_=line.split();mem[k.rstrip(":")+"_KiB"]=int(n)
    except OSError:pass
    gpu=[]
    try:
        p=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free",
          "--format=csv,noheader,nounits"],capture_output=True,text=True,timeout=5)
        if p.returncode==0:gpu=[line.strip() for line in p.stdout.splitlines() if line.strip()]
    except (OSError,subprocess.TimeoutExpired):pass
    disk=shutil.disk_usage(TEMP)
    return {"utc":datetime.now(timezone.utc).isoformat(),
            "runtime":"Kaggle" if KAGGLE else "local","python":sys.version.split()[0],
            "os":platform.platform(),"cpu_logical_cores":os.cpu_count(),
            "memory":mem,"packages":pkg,"gpus_observed":gpu,
            "scratch_disk_total_bytes":disk.total,"scratch_disk_free_bytes":disk.free}

def peak_rss():
    try:
        import resource
        n=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return round(n/(1024 if platform.system()=="Linux" else 1024*1024),2)
    except (OSError,ImportError):return None

def safe_write(filename,obj):
    path=OUT/filename;partial=OUT/(filename+".partial")
    partial.write_text(json.dumps(obj,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
    partial.replace(path)

def main():
    start=time.perf_counter()
    test=selftest()
    env=environment()
    safe_write("environment.json",env)
    dsu=DSU()
    records_by_source={};raw_by_source={}
    strict_by_source={};body_by_source={};coarse_by_source={}
    key_first_parent={}
    comparable_text_labels=defaultdict(set)  # D2/D3 three-class candidate taxonomy only
    parent_stats=defaultdict(lambda:{"rows":0,"files":set(),"family":None})
    source_stats={}
    timings={}
    errors={}
    downloaded=0
    for config in SOURCES:
        name,family,fn,sha,nbytes,kind=config
        t=time.perf_counter()
        entry={"family":family,"expected_sha256":sha,"source_revision":
            {"zenodo":"8339691","github2":REV2,"github3":REV3,"hf":REV4}[kind],
            "status":"UNAVAILABLE"}
        try:
            path,mode,n_download=retrieve(config)
            downloaded+=n_download
            entry.update({"file_bytes":path.stat().st_size,
                          "sha256":sha,"access_mode":mode})
            counter=Counter();body_counter=Counter();coarse_counter=Counter()
            class_counts=Counter();parents_found=0;missing_text=0;rows=0
            # Streaming CSV and one-source-at-a-time JSON avoids concatenating raw data.
            for i,row in enumerate(rows_for(path),1):
                rows+=1
                if rows>250_000:raise ValueError("UNEXPECTED_ROW_CAP_EXCEEDED")
                cat=category(row,family);class_counts[cat]+=1
                parent,has_parent=parent_key(row,name,i)
                parents_found+=int(has_parent)
                info=parent_stats[parent]
                info["rows"]+=1;info["files"].add(name);info["family"]=family
                dsu.find(parent)
                strict,body,coarse=text_keys(row)
                if strict is None:
                    missing_text+=1;continue
                counter[strict]+=1
                if family in ("D2","D3") and cat in ("Phishing","Spam","Valid"):
                    comparable_text_labels[strict].add(cat)
                if body:body_counter[body]+=1
                if coarse:coarse_counter[coarse]+=1
                old=key_first_parent.setdefault(strict,parent)
                if old!=parent:dsu.union(old,parent)
            if rows==0:raise ValueError("ZERO_RECORDS")
            strict_by_source[name]=counter
            body_by_source[name]=body_counter
            coarse_by_source[name]=coarse_counter
            raw_by_source[name]=rows
            entry.update({"status":"PASS_DESCRIPTIVE","rows":rows,
                "candidate_label_counts":dict(class_counts),
                "missing_subject_and_body_or_content":missing_text,
                "rows_with_explicit_parent_identifier":parents_found,
                "unique_strict_text_keys":len(counter),
                "extra_exact_normalized_rows":sum(v-1 for v in counter.values() if v>1),
                "unique_body_only_keys":len(body_counter),
                "unique_coarse_template_keys":len(coarse_counter)})
            print(name,":",rows,"rows | strict-key groups",len(counter))
        except (OSError,ValueError,UnicodeError,MemoryError,csv.Error,
                json.JSONDecodeError,TimeoutError) as e:
            entry.update({"status":"FAILED_OR_UNAVAILABLE","error_type":type(e).__name__})
            errors[name]=type(e).__name__
            print(name,": FAILED",type(e).__name__)
        source_stats[name]=entry
        timings[name]=round(time.perf_counter()-t,3)
    complete=all(v["status"]=="PASS_DESCRIPTIVE" for v in source_stats.values())
    pairs=[]
    for a,b in combinations(sorted(strict_by_source),2):
        shared=set(strict_by_source[a]) & set(strict_by_source[b])
        # Candidate-only weaker matches are reported separately; they are NEVER merged.
        b_shared=set(body_by_source[a]) & set(body_by_source[b])
        c_shared=set(coarse_by_source[a]) & set(coarse_by_source[b])
        same_family=source_stats[a]["family"]==source_stats[b]["family"]
        pairs.append({"source_a":a,"source_b":b,"same_dataset_family":same_family,
          "shared_strict_unique_text_keys":len(shared),
          "source_a_rows_in_shared_keys":sum(strict_by_source[a][k] for k in shared),
          "source_b_rows_in_shared_keys":sum(strict_by_source[b][k] for k in shared),
          "shared_body_only_candidate_keys":len(b_shared),
          "shared_coarse_template_candidate_keys":len(c_shared)})
    # Parent-aware DSU components from strict text matches, but never export identifiers.
    components=defaultdict(lambda:{"rows":0,"parents":0,"families":set(),"files":set()})
    for parent,info in parent_stats.items():
        c=components[dsu.find(parent)]
        c["rows"]+=info["rows"];c["parents"]+=1
        c["families"].add(info["family"]);c["files"].update(info["files"])
    comps=list(components.values())
    result={
      "release":RELEASE,"utc":datetime.now(timezone.utc).isoformat(),
      "status":"GLOBAL_DESCRIPTIVE_AUDIT_COMPLETE" if complete else "PARTIAL_SOURCE_FAILURE",
      "synthetic_selftest":test,"expected_sources":len(SOURCES),
      "sources_passed":sum(x["status"]=="PASS_DESCRIPTIVE" for x in source_stats.values()),
      "source_audits":source_stats,
      "pairwise_overlap":pairs,
      "global_lineage_components":{
        "total_exact_match_parent_components":len(comps),
        "components_with_multiple_source_files":sum(len(c["files"])>1 for c in comps),
        "components_spanning_D1_D2_D3_D4_families":sum(len(c["families"])>1 for c in comps),
        "rows_in_multi_file_components":sum(c["rows"] for c in comps if len(c["files"])>1),
        "largest_component_rows":max((c["rows"] for c in comps),default=0),
        "three_class_candidate_conflicting_strict_text_keys":sum(len(v)>1 for v in comparable_text_labels.values()),
        "component_size_distribution":dict(Counter(
             "1" if c["rows"]==1 else "2-9" if c["rows"]<10 else
             "10-99" if c["rows"]<100 else "100+" for c in comps))},
      "source_selection":{"D1":"five independently named constituent CSVs",
       "D2":"original plus three rephrase sets (categorized mirror intentionally excluded)",
       "D3":"original seeds plus 6-per-parent variants",
       "D4":"Chinese and English candidate content",
       "D5":"excluded: phishing-only AI-vs-human feature archive, not primary email text"},
      "split_freeze_gate":"NO_GO_PENDING_RIGHTS_LABELS_PARENT_REVIEW_AND_HOLDOUT_APPROVAL",
      "binary_model_training_gate":"NO_GO",
      "warnings":[
        "Exact cross-source text matching plus body-only and coarse candidate fingerprints are NOT exhaustive semantic/campaign deduplication.",
        "Coarse template overlap replaces URLs, numbers and emails; this is only a review candidate, never an automatic merge or label change.",
        "Three-class D2/D3 candidate labels must NOT be equated to 0/1 in D1/D4 until official semantics are verified.",
        "Any three-class candidate label conflicts reported here are annotation discrepancies, NOT independently adjudicated gold labels.",
        "No split manifests, raw emails, per-row hashes, join IDs or private annotations are exported.",
        "D2 No. and D3 Original_ID indicate candidate families but human provenance and semantic invariance remain pending.",
        "Same normalized text across sources suggests contamination risk; it does not automatically establish independent source lineage."
      ]}
    safe_write("lineage_audit.json",result)
    resource={
        "total_seconds":round(time.perf_counter()-start,3),
        "source_seconds":timings,
        "peak_process_rss_mb":peak_rss(),
        "download_bytes":downloaded,
        "scratch_disk_free_bytes_after":shutil.disk_usage(TEMP).free,
        "gpu_training_performed":False,"gpu_training_hours":0,
        "unique_parent_ids_in_memory":len(parent_stats),
        "unique_strict_keys_in_memory":len(key_first_parent)}
    safe_write("resource_usage.json",resource)
    manifest={"release":RELEASE,
       "source_sha256_by_id":{k:v["expected_sha256"] for k,v in source_stats.items()},
       "source_revision":{"D1":"Zenodo 8339691","D2":REV2,"D3":REV3,"D4":REV4},
       "output_files":["lineage_audit.json","environment.json","resource_usage.json","run_manifest.json","summary.txt"],
       "notebook_source_revision":"This notebook's GitHub commit must be recorded externally on execution review",
       "scientific_holdout_split_is_frozen":False}
    safe_write("run_manifest.json",manifest)
    line=["NOTEBOOK 04 — GLOBAL LEAKAGE AUDIT",
          "STATUS: "+result["status"],"Selftest: "+test,
          "Verified sources: "+str(result["sources_passed"])+"/"+str(len(SOURCES)),
          "Total rows scanned: "+str(sum(raw_by_source.values())),
          "Parent components: "+str(len(comps)),
          "Multi-source components: "+str(result["global_lineage_components"]["components_with_multiple_source_files"]),
          "Cross-family components: "+str(result["global_lineage_components"]["components_spanning_D1_D2_D3_D4_families"]),
          "Three-class strict-text label-conflict candidates: "+str(result["global_lineage_components"]["three_class_candidate_conflicting_strict_text_keys"]),
          "Elapsed seconds: "+str(resource["total_seconds"]),
          "Peak RSS MB: "+str(resource["peak_process_rss_mb"]),
          "Downloaded bytes: "+str(downloaded),
          "SPLIT FREEZE: NO-GO (rights/labels/pair approval pending)",
          "TRAINING: NO-GO",
          "Safe reports in "+str(OUT),
          "No raw emails, text hashes, parent IDs, unsafe URLs or private decisions published."]
    (OUT/"summary.txt").write_text("\n".join(line)+"\n",encoding="utf-8")
    print("\n"+"\n".join(line))
    return result

GLOBAL_LINEAGE=main()

## Output handoff

After Run All, share the safe folder `/kaggle/working/phishing_global_lineage_evidence/`, especially `lineage_audit.json`, `environment.json`, `resource_usage.json`, `run_manifest.json` and `summary.txt`. **Do not** attach raw email files, individual text keys, private annotations or live phishing links.

Only mark Notebook 04's **descriptive global-overlap audit** complete if all 13 sources verify. The **frozen split / model-training gate** remains blocked until rights, source-grounded phishing labels, original↔rewrite semantic validation and split design are approved separately.